In [2]:
# 1) Create a folder for data (optional but tidy)
!mkdir -p /content/data

# 2) Download the tar.gz file to that folder
!wget -O /content/data/aclImdb_v1.tar.gz \
  http://d2l-data.s3-accelerate.amazonaws.com/aclImdb_v1.tar.gz

# 3) Extract the tar.gz into /content/data
!tar -zxf /content/data/aclImdb_v1.tar.gz -C /content/data

--2025-12-26 07:32:50--  http://d2l-data.s3-accelerate.amazonaws.com/aclImdb_v1.tar.gz
Resolving d2l-data.s3-accelerate.amazonaws.com (d2l-data.s3-accelerate.amazonaws.com)... 13.249.183.82
Connecting to d2l-data.s3-accelerate.amazonaws.com (d2l-data.s3-accelerate.amazonaws.com)|13.249.183.82|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 84125825 (80M) [application/x-gzip]
Saving to: ‘/content/data/aclImdb_v1.tar.gz’

/content/data/aclIm 100%[===================>]  80.23M   104MB/s    in 0.8s    

2025-12-26 07:32:51 (104 MB/s) - ‘/content/data/aclImdb_v1.tar.gz’ saved [84125825/84125825]



In [3]:
!ls /content/data
!ls /content/data/aclImdb

aclImdb  aclImdb_v1.tar.gz
imdbEr.txt  imdb.vocab	README	test  train


In [4]:
# ==========================================
# Train a BiLSTM sentiment classifier on IMDb
# and save:
#   - model weights (state_dict)
#   - vocabulary and config
# ==========================================

import os               # For directory and path operations 
import re               # For regular expression based text cleaning 
from collections import Counter  # For counting token frequencies 

import torch            # Core PyTorch tensor & autograd library 
from torch import nn    # Neural network modules: Embedding, LSTM, Linear, etc. 
from torch.utils.data import Dataset, DataLoader  # Dataset + mini‑batch loader 

# -----------------------------
# 1. Hyperparameters
# -----------------------------
# These values control model capacity and training behavior 

max_len = 200      # Maximum number of tokens per review after truncation/padding.
                   # So each input sequence length T = 200 time steps .

min_freq = 5       # Only words that appear >= min_freq times in training data
                   # are kept in the vocabulary (others become <unk>) .

embed_size = 100   # Dimension of word embeddings: each token index is mapped
                   # to a 100‑dimensional real‑valued vector .

num_hiddens = 128  # Hidden size per direction of the LSTM; the BiLSTM has
                   # 2 * num_hiddens = 256 hidden units per time step .

num_layers = 1     # Number of stacked LSTM layers (depth of recurrent network) .

batch_size = 64    # Number of sequences per mini‑batch during training/testing .

lr = 1e-3          # Learning rate for Adam optimizer; controls step size in
                   # parameter updates during gradient descent .

num_epochs = 5     # Number of passes over the entire training dataset.
                   # Increase for better accuracy but longer training time .

# Use GPU if available; otherwise fall back to CPU .
device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# 2. Load IMDb data
# -----------------------------
# Folder structure assumed:
#   aclImdb/train/neg, aclImdb/train/pos
#   aclImdb/test/neg,  aclImdb/test/pos
# Each .txt file contains one review .

def read_imdb(data_dir):
    """
    Read all review files under data_dir ('aclImdb/train' or 'aclImdb/test').

    Returns:
        texts  : list of list of tokens (each inner list is one review)
        labels : list of ints (0 for negative, 1 for positive) 
    """
    texts = []   # Stores tokenized reviews
    labels = []  # Stores corresponding sentiment labels

    # Loop over both sentiment subfolders
    for label_type in ["neg", "pos"]:
        folder = os.path.join(data_dir, label_type)  # e.g. 'aclImdb/train/neg'

        # Iterate over all files in the folder
        for fname in os.listdir(folder):
            # Only use .txt files; ignore other possible files .
            if not fname.endswith(".txt"):
                continue

            # Open and read the entire review, convert to lowercase
            with open(os.path.join(folder, fname), encoding="utf-8") as f:
                text = f.read().lower()

            # Text cleaning:
            #   - Keep only characters a‑z, replace other characters
            #     (digits, punctuation, etc.) with spaces.
            #   - This simplifies tokenization and vocabulary .
            text = re.sub(r"[^a-z]+", " ", text)

            # Tokenization: split on whitespace -> list of word tokens .
            tokens = text.split()

            # Append tokens for this review
            texts.append(tokens)

            # Map folder name to sentiment label
            #   'neg' -> 0, 'pos' -> 1 (binary classification) .
            labels.append(0 if label_type == "neg" else 1)

    return texts, labels

# Read training and test sets .
train_texts, train_labels = read_imdb("data/aclImdb/train")
test_texts, test_labels = read_imdb("data/aclImdb/test")

# -----------------------------
# 3. Build vocabulary and encode reviews
# -----------------------------

# Flatten all tokens from training reviews and count their frequencies.
# This gives a mapping token -> frequency .
counter = Counter([w for text in train_texts for w in text])

# Initialize vocabulary with two special tokens:
#   <pad>: used for padding shorter sequences (index 0)
#   <unk>: used for out‑of‑vocabulary tokens (index 1) .
vocab = {"<pad>": 0, "<unk>": 1}

# Add words that appear at least min_freq times to the vocabulary .
for word, freq in counter.items():
    if freq >= min_freq:
        # Assign next available integer index to this word
        vocab[word] = len(vocab)

# Total vocabulary size (including special tokens) .
vocab_size = len(vocab)


def encode(text_tokens):
    """
    Convert a list of tokens into a fixed‑length list of integer indices.

    Steps:
      1. Map each token to its index using vocab; unknown tokens -> index 1 (<unk>).
      2. Truncate the list to max_len if it is too long.
      3. Pad with index 0 (<pad>) at the end if it is shorter than max_len.

    Result:
      - A list of length exactly max_len.
      - This makes every review a sequence of exactly T = max_len time steps, so
        RNN inputs have uniform length .
    """
    # Map tokens to indices; default to 1 (<unk>) if token not in vocab .
    ids = [vocab.get(w, 1) for w in text_tokens][:max_len]

    # Pad with zeros (index of <pad>) so sequence length = max_len .
    if len(ids) < max_len:
        ids += [0] * (max_len - len(ids))

    return ids


# Encode all reviews into sequences of indices (length = max_len) .
train_ids = [encode(t) for t in train_texts]
test_ids = [encode(t) for t in test_texts]

# -----------------------------
# 4. Dataset and DataLoader
# -----------------------------

class IMDBDataset(Dataset):
    """
    Custom PyTorch Dataset:
    - Stores encoded sequences (lists of indices) and labels.
    - __getitem__ returns one encoded review and its sentiment label .
    """

    def __init__(self, ids, labels):
        self.ids = ids        # List of encoded sequences
        self.labels = labels  # List of integer labels

    def __len__(self):
        # Total number of samples in this dataset .
        return len(self.ids)

    def __getitem__(self, idx):
        """
        Returns:
            x: LongTensor of shape (max_len,) representing token indices.
            y: LongTensor scalar with the class label (0 or 1) .
        """
        x = torch.tensor(self.ids[idx], dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

# Create DataLoaders that generate mini‑batches .
train_loader = DataLoader(IMDBDataset(train_ids, train_labels),
                          batch_size=batch_size,
                          shuffle=True)      # Shuffle for SGD

test_loader = DataLoader(IMDBDataset(test_ids, test_labels),
                         batch_size=batch_size,
                         shuffle=False)     # No shuffle needed when evaluating

# -----------------------------
# 5. Bi‑directional LSTM model
# -----------------------------
# Architecture matches the BiRNN description in your document:
#   Embedding -> BiLSTM -> concatenate states -> Linear classifier .

class BiRNN(nn.Module):
    """
    Bidirectional LSTM for text sentiment classification.

    Mathematical view:
    - Input sequence x_1, ..., x_T where x_t is an embedding vector.
    - A forward LSTM computes hidden states h_t^f from t = 1..T.
    - A backward LSTM computes hidden states h_t^b from t = T..1.
      Each time step has concatenated state h_t = [h_t^f ; h_t^b] in R^{2H}. 
    - We take h_1 and h_T and concatenate: e = [h_1 ; h_T] in R^{4H}. 
    - The Linear layer computes logits z = We + b in R^2.
      Cross‑EntropyLoss then uses softmax(z) to compute class probabilities .
    """

    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers):
        super(BiRNN, self).__init__()

        # Embedding layer:
        # Input: LongTensor of indices with shape (batch_size, seq_len).
        # Output: FloatTensor with shape (batch_size, seq_len, embed_size).
        # Each index i is mapped to vector e_i in R^{embed_size} .
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # Bi‑directional LSTM encoder:
        # input_size  = embed_size: dimension of embedding vectors.
        # hidden_size = num_hiddens: hidden dimension H per direction.
        # num_layers  = num_layers: stacked LSTM layers (depth).
        # bidirectional = True: we get both forward and backward states,
        # so output dimension per time step is 2 * num_hiddens .
        self.encoder = nn.LSTM(embed_size,
                               num_hiddens,
                               num_layers=num_layers,
                               bidirectional=True)

        # Fully connected decoder:
        # Input features = 4 * num_hiddens (concatenation of h_1 and h_T),
        # Output features = 2 (negative vs positive class) .
        self.decoder = nn.Linear(4 * num_hiddens, 2)

    def forward(self, inputs):
        """
        Args:
            inputs: LongTensor of shape (batch_size, max_len)
                    containing token indices.

        Returns:
            outs: FloatTensor of shape (batch_size, 2)
                  containing unnormalized logits for two classes.

        Shape tracking:
        - inputs: (B, T)
        - embedding: (B, T, E)
        - after permute: (T, B, E)
        - LSTM outputs: (T, B, 2H)
        - first_step / last_step: (B, 2H)
        - encoding concat: (B, 4H)
        - decoder output: (B, 2)  
        """
        # Convert indices to embeddings:
        # embeddings[b, t, :] is the vector for the t‑th token in the b‑th sample.
        embeddings = self.embedding(inputs)          # (B, T, E)

        # PyTorch LSTM expects input shape (seq_len, batch, input_size),
        # so we permute dimensions: (B, T, E) -> (T, B, E) .
        embeddings = embeddings.permute(1, 0, 2)     # (T, B, E)

        # Run the sequence through the BiLSTM.
        # outputs[t, b, :] = concatenated hidden state at time t (forward+backward)
        #  in R^{2H}. H = num_hiddens. 
        # The second output (hidden, cell) is unused in this architecture.
        outputs, _ = self.encoder(embeddings)        # (T, B, 2H)

        # Take hidden state for the first time step (t = 0).
        # This corresponds to information near the beginning of the sequence.
        first_step = outputs[0]                      # (B, 2H)

        # Take hidden state for the last time step (t = T‑1).
        # This corresponds to information near the end of the sequence.
        last_step = outputs[-1]                      # (B, 2H)

        # Concatenate first and last hidden states:
        # encoding[b] = [h_1(b); h_T(b)] in R^{4H} .
        encoding = torch.cat((first_step, last_step), dim=1)  # (B, 4H)

        # Linear layer computes logits:
        #   z = W * encoding^T + b
        # for each sample, where W in R^{2 x 4H}, b in R^2.
        outs = self.decoder(encoding)                # (B, 2)

        # We return logits (not softmax). CrossEntropyLoss will apply
        # softmax internally when computing the loss. 
        return outs

# Instantiate model and move it to the chosen device .
net = BiRNN(vocab_size, embed_size, num_hiddens, num_layers).to(device)

# -----------------------------
# 6. Loss function and optimizer
# -----------------------------

# Cross‑entropy loss for multi‑class classification.
# Given logits z (B, 2) and labels y (B,), it computes:
#   p_k = exp(z_k) / sum_j exp(z_j)
#   L = − (1/B) * sum_i log p_{y_i}  .
criterion = nn.CrossEntropyLoss()

# Adam optimizer updates parameters using adaptive learning rates and
# running estimates of first and second moments of gradients .
optimizer = torch.optim.Adam(net.parameters(), lr=lr)

# -----------------------------
# 7. Training loop
# -----------------------------

for epoch in range(num_epochs):
    net.train()  # Enable training mode (e.g., dropout if present).

    total_loss = 0.0     # Sum of losses over all samples in this epoch.
    total_correct = 0    # Number of correct predictions.
    total_examples = 0   # Number of samples processed.

    # Iterate over mini‑batches from the training loader .
    for X, y in train_loader:
        # X: (B, T) indices; y: (B,) labels.
        X, y = X.to(device), y.to(device)

        # Forward pass: compute logits for this batch.
        outputs = net(X)              # outputs: (B, 2)

        # Compute cross‑entropy loss for this batch.
        loss = criterion(outputs, y)  # scalar tensor

        # Zero any existing gradients stored in optimizer.
        optimizer.zero_grad()

        # Backward pass: compute gradients dL/dθ for all parameters θ.
        loss.backward()

        # Parameter update:
        #   θ_new = θ_old − lr * (Adam‑adjusted gradient) .
        optimizer.step()

        # Accumulate statistics for reporting epoch averages.
        batch_size_actual = X.size(0)
        total_loss += loss.item() * batch_size_actual

        # Predicted class = argmax over logits along class dimension.
        preds = outputs.argmax(dim=1)          # (B,)

        # Count correct predictions.
        total_correct += (preds == y).sum().item()

        # Accumulate total number of samples.
        total_examples += batch_size_actual

    # Compute average loss and accuracy over the epoch.
    train_loss = total_loss / total_examples
    train_acc = total_correct / total_examples
    print(f"Epoch {epoch+1}: loss={train_loss:.4f}, acc={train_acc:.4f}")

# -----------------------------
# 8. Evaluate on test set
# -----------------------------

net.eval()  # Switch to evaluation mode.

test_correct = 0
test_examples = 0

# In evaluation, gradients are not needed; disable them for efficiency.
with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        outputs = net(X)                 # (B, 2)
        preds = outputs.argmax(dim=1)    # (B,)
        test_correct += (preds == y).sum().item()
        test_examples += X.size(0)

# Test accuracy = (#correct predictions) / (total test samples).
test_acc = test_correct / test_examples
print(f"Test accuracy: {test_acc:.4f}")

# -----------------------------
# 9. Save trained model and vocabulary
# -----------------------------

# Ensure directory for model files exists.
os.makedirs("saved_models", exist_ok=True)

# Save only the learned parameters (state_dict).
# This is recommended over saving the entire model object .
model_path = "saved_models/imdb_birnn.pth"
torch.save(net.state_dict(), model_path)

# Save vocabulary and important config to reconstruct model for testing.
vocab_path = "saved_models/imdb_vocab.pt"
torch.save({
    "vocab": vocab,
    "max_len": max_len,
    "embed_size": embed_size,
    "num_hiddens": num_hiddens,
    "num_layers": num_layers
}, vocab_path)

print(f"Saved model to {model_path}")
print(f"Saved vocab/config to {vocab_path}")


Epoch 1: loss=0.6760, acc=0.5730
Epoch 2: loss=0.6054, acc=0.6801
Epoch 3: loss=0.4872, acc=0.7693
Epoch 4: loss=0.3378, acc=0.8580
Epoch 5: loss=0.2364, acc=0.9082
Test accuracy: 0.8153
Saved model to saved_models/imdb_birnn.pth
Saved vocab/config to saved_models/imdb_vocab.pt


In [5]:
# ==========================================
# Load the trained BiLSTM sentiment model and
# run inference on example reviews.
# ==========================================

import re          # For cleaning new input text 
import torch
from torch import nn

# Use same device logic as training.
device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# 1. Load vocabulary and config
# -----------------------------

# Load saved vocab and model hyperparameters created during training.
# This ensures that encoding of new text matches training exactly .
vocab_data = torch.load("saved_models/imdb_vocab.pt", map_location=device)

vocab = vocab_data["vocab"]          # Token -> index mapping
max_len = vocab_data["max_len"]      # Must match training 
embed_size = vocab_data["embed_size"]
num_hiddens = vocab_data["num_hiddens"]
num_layers = vocab_data["num_layers"]
vocab_size = len(vocab)              # Total number of entries in vocab

# -----------------------------
# 2. Define the same BiRNN model
# -----------------------------
# The architecture must be identical to the one used in training,
# otherwise state_dict loading will fail or produce incorrect behavior .

class BiRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers):
        super(BiRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.encoder = nn.LSTM(embed_size, num_hiddens,
                               num_layers=num_layers,
                               bidirectional=True)
        self.decoder = nn.Linear(4 * num_hiddens, 2)

    def forward(self, inputs):
        """
        Same forward computation as in training:

        - inputs: (B, T)
        - embeddings: (T, B, E)
        - LSTM outputs: (T, B, 2H)
        - encoding: (B, 4H)
        - outs: (B, 2)
        """
        emb = self.embedding(inputs)         # (B, T, E)
        emb = emb.permute(1, 0, 2)           # (T, B, E)
        outputs, _ = self.encoder(emb)       # (T, B, 2H)
        first_step = outputs[0]              # (B, 2H)
        last_step = outputs[-1]              # (B, 2H)
        encoding = torch.cat((first_step, last_step), dim=1)  # (B, 4H)
        outs = self.decoder(encoding)        # (B, 2)
        return outs

# Instantiate model and load state dictionary (weights).
net = BiRNN(vocab_size, embed_size, num_hiddens, num_layers).to(device)
state_dict = torch.load("saved_models/imdb_birnn.pth", map_location=device)
net.load_state_dict(state_dict)
net.eval()  # Evaluation mode (no dropout, etc.)

# -----------------------------
# 3. Helper functions for new text
# -----------------------------

def clean_and_tokenize(text):
    """
    Clean and tokenize an input string in the same way as during training:

    - Convert to lowercase.
    - Replace all non‑a‑z characters with spaces using regex.
    - Split on whitespace to get tokens.

    This consistency is crucial so that the learned model sees text in the
    same format at inference time as it did during training .
    """
    text = text.lower()
    text = re.sub(r"[^a-z]+", " ", text)
    tokens = text.split()
    return tokens

def encode(tokens):
    """
    Encode a list of tokens into a fixed‑length list of indices:

    - Each token is mapped using vocab; unknown tokens -> index of <unk>.
    - Sequence is truncated to max_len or padded with <pad> index to reach
      length max_len.

    Output:
        ids: list[int] of length max_len .
    """
    ids = [vocab.get(w, 1) for w in tokens][:max_len]
    if len(ids) < max_len:
        ids += [0] * (max_len - len(ids))
    return ids

# -----------------------------
# 4. Single‑sentence prediction function
# -----------------------------

def predict_sentiment(text):
    """
    Predict sentiment for a single review string.

    Workflow:
      1. Clean and tokenize text -> tokens.
      2. Encode tokens into indices of length max_len.
      3. Wrap into a batch of size 1: shape (1, max_len).
      4. Forward pass through the model to get logits z in R^2.
      5. Apply softmax to get probabilities p_neg, p_pos.
      6. Choose the label with maximum probability.

    Returns:
      pred_label: string "positive" or "negative".
      prob_neg  : probability for negative class.
      prob_pos  : probability for positive class.
    """
    # Step 1‑2: preprocessing consistent with training.
    tokens = clean_and_tokenize(text)
    ids = encode(tokens)

    # Convert to tensor and add batch dimension.
    # X shape: (1, max_len), type: long (indices) .
    X = torch.tensor([ids], dtype=torch.long).to(device)

    with torch.no_grad():  # Disable gradient computation for inference.
        # Forward pass: compute logits for the single sample.
        outputs = net(X)                       # (1, 2)

        # Softmax converts logits z into probabilities:
        #   p_k = exp(z_k) / sum_j exp(z_j).
        probs = torch.softmax(outputs, dim=1)  # (1, 2)

        # Extract scalar probabilities for each class.
        prob_neg = probs[0, 0].item()
        prob_pos = probs[0, 1].item()

        # Choose class with higher probability.
        pred_label = "positive" if prob_pos >= prob_neg else "negative"

    return pred_label, prob_neg, prob_pos

# -----------------------------
# 5. Example predictions
# -----------------------------

if __name__ == "__main__":
    # Example 1: clearly negative review.
    text1 = "This movie is terrible. The plot is boring and the acting is bad."
    pred1, pneg1, ppos1 = predict_sentiment(text1)
    print(f"Text 1: {text1}")
    print(f"Prediction: {pred1} (P(neg)={pneg1:.3f}, P(pos)={ppos1:.3f})")

    # Example 2: clearly positive review.
    text2 = "I really loved this movie. The story was touching and the actors were amazing."
    pred2, pneg2, ppos2 = predict_sentiment(text2)
    print(f"\nText 2: {text2}")
    print(f"Prediction: {pred2} (P(neg)={pneg2:.3f}, P(pos)={ppos2:.3f})")


Text 1: This movie is terrible. The plot is boring and the acting is bad.
Prediction: negative (P(neg)=0.997, P(pos)=0.003)

Text 2: I really loved this movie. The story was touching and the actors were amazing.
Prediction: positive (P(neg)=0.010, P(pos)=0.990)
